# A DQN that learns from rewards — Rainbow-lite on CartPole

_Capstones_

---

This notebook is a practice scaffold for the **A DQN that learns from rewards — Rainbow-lite on CartPole** lesson. We build the agent one improvement at a time — the base DQN loop, the Double-DQN target, the Dueling head, and Prioritized replay — then race the combined agent against a vanilla baseline.

The central idea is the **DQN learning update**: store experience, read `Q(s,a)`, build a bootstrap target, take a gradient step, and periodically copy the online network into a target network. We will also unpack the **dueling aggregation** `Q = V + (A - mean(A))` tensor by tensor, because that is the easiest Rainbow-lite component to see directly.

Cells marked **🎛️ Play with it** are interactive sandboxes: in Colab they show sliders and dropdowns you can drag; without widgets they still render a sensible default. Run each cell top to bottom, experiment with the knobs, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / matplotlib ship with Colab. Gymnasium does not.
!pip install -q gymnasium
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## First, look at the data

CartPole is not a fixed dataset. The "data" is a stream of transitions generated by acting in an environment: `(state, action, reward, next_state, done)`. Before building a model, inspect the four observation numbers, the two actions, a random-policy baseline, and a tiny replay table.

### Step 1 — Import RL/PyTorch tools and pin the seeds

The original scaffold used these seeds; keep them fixed so our run is reproducible. `GAMMA = 0.99` means rewards roughly one hundred steps in the future still matter.

In [ ]:
# In Colab first run:  !pip install gymnasium
# (torch is preinstalled in Colab; gymnasium is not.)
import random
import torch
import torch.nn as nn
import gymnasium as gym

torch.manual_seed(0)
random.seed(0)
np.random.seed(0)
GAMMA = 0.99

### Step 2 — What does CartPole give us?

Each observation is a 4-number summary of the cart and pole. Each action is a push left or right. This table is our schema: it tells us what the neural network will read and what it must choose between.

In [ ]:
env = gym.make("CartPole-v1")
obs, _ = env.reset(seed=0)
obs_df = pd.DataFrame({
    "index": [0, 1, 2, 3],
    "observation dimension": ["cart position", "cart velocity", "pole angle", "pole angular velocity"],
    "current value": [float(x) for x in obs],
    "intuition": ["left/right location", "left/right speed", "tilt from vertical", "tilt speed"],
})
action_df = pd.DataFrame({
    "action id": [0, 1],
    "meaning": ["push cart left", "push cart right"],
})
print("observation shape:", env.observation_space.shape, " number of actions:", env.action_space.n)
display(obs_df)
display(action_df)
env.close()

### Step 3 — A random-policy baseline

A random policy ignores the state and picks left/right uniformly. CartPole gives +1 reward per alive timestep, so the episode return is exactly the number of timesteps survived. A DQN must beat this baseline.

In [ ]:
def run_random_policy(n_episodes=5):
    env = gym.make("CartPole-v1")
    rows = []
    for ep in range(n_episodes):
        s, _ = env.reset(seed=ep)
        done, total, steps = False, 0.0, 0
        while not done:
            a = env.action_space.sample()
            s, r, term, trunc, _ = env.step(a)
            done = term or trunc
            total += r
            steps += 1
        rows.append({"episode": ep, "return": total, "steps": steps})
    env.close()
    return pd.DataFrame(rows)

random_baseline = run_random_policy(5)
display(random_baseline)
print("random-policy mean return (our run):", round(float(random_baseline["return"].mean()), 1))
plt.figure(figsize=(6, 3))
plt.bar(random_baseline["episode"], random_baseline["return"], color="#79c0ff")
plt.axhline(random_baseline["return"].mean(), color="#ff6b6b", ls="--", label="mean")
plt.xlabel("episode"); plt.ylabel("return")
plt.title("random policy baseline")
plt.legend(); plt.show()

### Step 4 — A replay buffer is a table of transitions

DQN learns from stored past experience instead of only the most recent step. Here are the first few random transitions in the exact shape the replay buffer will store: state `s`, action `a`, reward `r`, next state `s'`, and terminal flag `done`.

In [ ]:
def collect_random_transitions(n=8, seed=1):
    env = gym.make("CartPole-v1")
    s, _ = env.reset(seed=seed)
    rows = []
    for t in range(n):
        a = env.action_space.sample()
        s2, r, term, trunc, _ = env.step(a)
        done = term or trunc
        rows.append({
            "t": t,
            "s": np.round(s, 3).tolist(),
            "a": int(a),
            "r": float(r),
            "s'": np.round(s2, 3).tolist(),
            "done": bool(done),
        })
        s = s2
        if done:
            s, _ = env.reset(seed=seed + t + 1)
    env.close()
    return pd.DataFrame(rows)

transition_df = collect_random_transitions()
display(transition_df)

**🎛️ Play with it — epsilon means exploration**

`epsilon` is the probability of ignoring the Q-network and taking a random exploratory action. **Try:** move epsilon from 1.0 to 0.0. **Watch:** random mass shrink and greedy mass grow. **Why it matters:** early exploration fills the replay buffer with diverse experience; later exploitation uses what the network learned.

In [ ]:
def play(epsilon=0.30):
    probs = pd.DataFrame({
        "choice type": ["random action", "greedy Q action"],
        "probability": [epsilon, 1.0 - epsilon],
    })
    display(probs)
    plt.figure(figsize=(5, 3))
    plt.bar(probs["choice type"], probs["probability"], color=["#ffb86b", "#7ee787"])
    plt.ylim(0, 1); plt.ylabel("probability")
    plt.title("epsilon-greedy mix")
    plt.show()

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox, Text
    interact(play, epsilon=FloatSlider(value=0.30, min=0.0, max=1.0, step=0.05))
except Exception:
    play()        # no ipywidgets (or non-notebook) -> render sensible defaults

**🎛️ Play with it — gamma sets the effective horizon**

The target adds future value as `gamma^t`. **Try:** lower and raise gamma. **Watch:** the discount-weight curve flatten as gamma approaches 1. **Why it matters:** CartPole rewards every timestep alive, so a high gamma encourages long balancing behavior.

In [ ]:
def play(gamma=0.99, steps=40):
    t = np.arange(steps)
    weights = gamma ** t
    horizon = float(weights.sum())
    plt.figure(figsize=(6, 3))
    plt.bar(t, weights, color="#79c0ff")
    plt.xlabel("future step t"); plt.ylabel("gamma^t")
    plt.title("discount weights")
    plt.show()
    print(f"sum of first {steps} weights = {horizon:.1f}; infinite-horizon sum = {1/(1-gamma):.1f}")

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox, Text
    interact(play,
             gamma=FloatSlider(value=0.99, min=0.50, max=0.999, step=0.01),
             steps=IntSlider(value=40, min=5, max=100, step=5))
except Exception:
    play()        # no ipywidgets (or non-notebook) -> render sensible defaults

## Reference implementation — gymnasium + PyTorch (Colab)

Rainbow combined six Atari improvements; our **Rainbow-lite** keeps the components that are compact enough for a teaching notebook: the base DQN loop, Double-DQN target, Dueling head, and Prioritized replay.

| Component | What it contributes | In our code |
|---|---|---|
| DQN | Q-learning with a neural net, replay, target net | `learn`, `train` |
| Double DQN | online net selects next action; target net evaluates it | `use_double=True` branch |
| Dueling DQN | split `Q` into value `V` and advantages `A` | `QNet(dueling=True)` |
| Prioritized replay | sample high-TD-error transitions more often | `PrioritizedReplay(per_on=True)` |

### Step 5 — Sanity-check the dueling formula

A dueling head splits a state into one scalar value `V(s)` and per-action advantages `A(s,a)`, then recombines them as `Q = V + (A - mean_a A)`. Subtracting the mean removes the redundancy between V and A so the two streams stay identifiable.

In [ ]:
# --- Sanity-check the DUELING head's Eq. 9 aggregation: Q = V + (A - mean_a A). ---
V = torch.tensor([[10.0]])                  # state-value V(s), shape [1,1]
A = torch.tensor([[6.0, 0.0]])              # advantage A(s,a) for two actions, shape [1,2]
mean_A = A.mean(dim=1, keepdim=True)        # mean advantage across actions = 3
Q = V + (A - mean_A)                        # Q = 10 + (6-3, 0-3) = [13, 7]
worked = pd.DataFrame({
    "action": ["left", "right"],
    "V(s)": [float(V.item()), float(V.item())],
    "A(s,a)": A.numpy()[0],
    "A - mean(A)": (A - mean_A).numpy()[0],
    "Q(s,a)": Q.numpy()[0],
})
display(worked)
plt.figure(figsize=(6, 3))
plt.bar(worked["action"], worked["Q(s,a)"], color="#c89bff")
plt.ylabel("Q value")
plt.title("dueling formula output")
plt.show()
print("dueling worked example:  V =", V.item(), " A =", A.tolist()[0],
      " mean(A) =", A.mean().item(), " Q = V+(A-mean) =", Q.tolist()[0])

### Step 6 — Build the Q-network

`QNet` is the Q-network (COMPONENT 3, Dueling DQN). A shared torso feeds either a single Q head (plain DQN) or a value stream plus an advantage stream that recombine with the Eq. 9 formula we just checked.

In [ ]:
# ============================================================================
# COMPONENT 3 (Dueling DQN): a Q-network whose head optionally splits into a
# value stream V(s) and an advantage stream A(s,a), recombined by Eq. 9.
# ============================================================================
class QNet(nn.Module):
    def __init__(self, obs_dim, n_act, hidden=128, dueling=True):
        super().__init__()
        self.dueling = dueling
        self.torso = nn.Sequential(nn.Linear(obs_dim, hidden), nn.ReLU(),
                                   nn.Linear(hidden, hidden), nn.ReLU())
        if dueling:
            self.value_head = nn.Linear(hidden, 1)        # V(s):    one scalar per state
            self.adv_head   = nn.Linear(hidden, n_act)    # A(s,a):  one number per action
        else:
            self.q_head = nn.Linear(hidden, n_act)        # plain DQN: Q(s,a) directly
    def forward(self, x):
        h = self.torso(x)
        if self.dueling:
            v = self.value_head(h)                        # [batch, 1]
            a = self.adv_head(h)                          # [batch, n_act]
            return v + (a - a.mean(dim=1, keepdim=True))  # Eq. 9 dueling aggregation
        return self.q_head(h)

### Step 7 — Inspect QNet shapes and parameters

A single CartPole state becomes two Q-values, one per action. The parameter table shows where the model capacity lives.

In [ ]:
torch.manual_seed(0)
demo_q = QNet(obs_dim=4, n_act=2, dueling=True)
demo_state = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
with torch.no_grad():
    demo_out = demo_q(demo_state)
shape_df = pd.DataFrame({
    "tensor": ["state batch", "Q(state, actions)"],
    "shape": [tuple(demo_state.shape), tuple(demo_out.shape)],
})
param_rows = []
for name, p in demo_q.named_parameters():
    group = name.split(".")[0]
    param_rows.append({"component": group, "parameter": name, "count": p.numel()})
param_df = pd.DataFrame(param_rows).groupby("component", as_index=False)["count"].sum()
param_df["% of total"] = (100 * param_df["count"] / param_df["count"].sum()).round(1)
display(shape_df)
display(param_df)
print("untrained Q values for one state:", np.round(demo_out.numpy()[0], 4).tolist())

### Step 8 — Build the replay buffer

`PrioritizedReplay` (COMPONENT 4) stores past transitions; when `per_on=True` it samples transitions in proportion to their TD-error priority and returns importance-sampling weights to correct the resulting bias. With `per_on=False` it degrades to the uniform buffer of the original DQN.

In [ ]:
# ============================================================================
# COMPONENT 4 (Prioritized Experience Replay): sample by TD-error priority,
# return importance-sampling (IS) weights. With per_on=False it is a plain
# uniform replay buffer (every transition equally likely, weights = 1).
# ============================================================================
class PrioritizedReplay:
    def __init__(self, capacity=50000, alpha=0.6, per_on=True):
        self.capacity = capacity
        self.alpha = alpha
        self.per_on = per_on
        self.buf = []
        self.prios = np.zeros((capacity,), dtype=np.float32)
        self.pos = 0
    def push(self, s, a, r, s2, done):
        max_p = self.prios.max() if self.buf else 1.0    # new transitions get max priority
        if len(self.buf) < self.capacity:
            self.buf.append((s, a, r, s2, done))
        else:
            self.buf[self.pos] = (s, a, r, s2, done)
        self.prios[self.pos] = max_p
        self.pos = (self.pos + 1) % self.capacity
    def sample(self, n, beta=0.4):
        N = len(self.buf)
        if self.per_on:
            prios = self.prios[:N] ** self.alpha          # p_i^alpha
            P = prios / prios.sum()                       # P(i) = p_i^a / sum_k p_k^a  (Eq. 1)
            idx = np.random.choice(N, n, p=P)
            w = (N * P[idx]) ** (-beta)                   # w_i = (1/(N*P(i)))^beta  (S 3.4)
            w = w / w.max()                               # normalize for stability
        else:
            idx = np.random.choice(N, n)                  # uniform: the original DQN buffer
            w = np.ones(n, dtype=np.float32)
        s, a, r, s2, d = zip(*[self.buf[i] for i in idx])
        return (torch.tensor(np.array(s), dtype=torch.float32),
                torch.tensor(a, dtype=torch.long),
                torch.tensor(r, dtype=torch.float32),
                torch.tensor(np.array(s2), dtype=torch.float32),
                torch.tensor(d, dtype=torch.float32),
                idx, torch.tensor(w, dtype=torch.float32))
    def update_priorities(self, idx, td_errors):
        for i, e in zip(idx, td_errors):
            self.prios[i] = abs(float(e)) + 1e-5          # p_i = |TD error| + epsilon
    def __len__(self):
        return len(self.buf)

### Step 9 — Inspect replay sampling probabilities

New transitions start with max priority so they are not ignored. After learning, TD errors refresh priorities. The table shows how priorities become sampling probabilities.

In [ ]:
small_replay = PrioritizedReplay(capacity=20, alpha=0.6, per_on=True)
for row in transition_df.to_dict("records"):
    small_replay.push(np.array(row["s"], dtype=np.float32), row["a"], row["r"],
                      np.array(row["s'"], dtype=np.float32), row["done"])
small_replay.update_priorities(np.arange(len(small_replay)), np.linspace(0.1, 2.0, len(small_replay)))
P = small_replay.prios[:len(small_replay)] ** small_replay.alpha
P = P / P.sum()
per_df = pd.DataFrame({"transition": np.arange(len(small_replay)),
                       "priority": small_replay.prios[:len(small_replay)],
                       "sample probability": P})
display(per_df.round(4))
plt.figure(figsize=(6, 3))
plt.bar(per_df["transition"], per_df["sample probability"], color="#ffb86b")
plt.xlabel("transition index"); plt.ylabel("P(sample)")
plt.title("prioritized replay probabilities")
plt.show()

**🎛️ Play with it — priorities and alpha**

Prioritized replay samples in proportion to `priority^alpha`. **Try:** set `alpha=0` and then raise it. **Watch:** equal probabilities become sharply focused on high-priority transitions. **Why it matters:** high-error examples are replayed more often, but not exclusively.

In [ ]:
def play(alpha=0.60):
    priorities = np.array([0.1, 0.2, 0.5, 1.0, 2.0, 4.0], dtype=float)
    probs = priorities ** alpha
    probs = probs / probs.sum()
    df = pd.DataFrame({"transition": range(len(priorities)), "priority": priorities, "P(sample)": probs})
    display(df.round(4))
    plt.figure(figsize=(6, 3))
    plt.bar(df["transition"], df["P(sample)"], color="#ffb86b")
    plt.ylim(0, max(0.5, probs.max() * 1.15))
    plt.xlabel("transition"); plt.ylabel("probability")
    plt.title("PER alpha")
    plt.show()

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox, Text
    interact(play, alpha=FloatSlider(value=0.60, min=0.0, max=1.5, step=0.1))
except Exception:
    play()        # no ipywidgets (or non-notebook) -> render sensible defaults

## The hero operation, built tensor-by-tensor

The learning update is where all components meet. We will follow one concrete transition: compute `Q(s,a)`, build the Double-DQN target, form the TD error, and then zoom into the dueling head that produced those Q-values.

### Step 10 — Start from one concrete transition

Pick transition 0 from our tiny replay table. It says: from state `s`, take action `a`, get reward `r`, land in `s'`, and maybe terminate. Everything in `learn()` is built from batches of rows like this.

In [ ]:
one = transition_df.iloc[0]
s = torch.tensor(one["s"], dtype=torch.float32).unsqueeze(0)
a = torch.tensor([one["a"]], dtype=torch.long)
r = torch.tensor([one["r"]], dtype=torch.float32)
s2 = torch.tensor(one["s'"], dtype=torch.float32).unsqueeze(0)
done = torch.tensor([float(one["done"])], dtype=torch.float32)
display(pd.DataFrame({
    "piece": ["s", "a", "r", "s'", "done"],
    "value": [np.round(s.numpy()[0], 3).tolist(), int(a.item()), float(r.item()),
              np.round(s2.numpy()[0], 3).tolist(), bool(done.item())],
}))

### Step 11 — Online Q-values and `Q(s,a)`

The online network scores both actions. The action actually taken selects one column with `gather`, giving the scalar `Q_online(s,a)` that will be nudged toward the target.

In [ ]:
torch.manual_seed(0)
online_q = QNet(4, 2, dueling=True)
target_q = QNet(4, 2, dueling=True)
target_q.load_state_dict(online_q.state_dict())
with torch.no_grad():
    q_values = online_q(s)
    q_sa = q_values.gather(1, a.unsqueeze(1)).squeeze(1)
q_df = pd.DataFrame({"action": ["left (0)", "right (1)"], "Q_online(s,a)": q_values.numpy()[0]})
display(q_df.round(4))
plt.figure(figsize=(5, 3))
plt.bar(q_df["action"], q_df["Q_online(s,a)"], color="#7ee787")
plt.axvline(int(a.item()), color="#ff6b6b", lw=3, alpha=0.5, label="taken action")
plt.ylabel("Q value")
plt.title("Q values for one state")
plt.legend(); plt.show()
print("selected Q(s,a):", round(float(q_sa.item()), 4))

### Step 12 — Double-DQN target: select with online, evaluate with target

Vanilla DQN uses `max_a Q_target(s',a)` for both action selection and evaluation. Double DQN splits those jobs: the online net picks `argmax_a Q_online(s',a)`, and the target net evaluates that selected action. Then `y = r + gamma * (1-done) * Q_target(s', selected_action)`.

In [ ]:
with torch.no_grad():
    next_online = online_q(s2)
    next_target = target_q(s2)
    next_a = next_online.argmax(1, keepdim=True)
    q_next_double = next_target.gather(1, next_a).squeeze(1)
    y_double = r + GAMMA * (1.0 - done) * q_next_double
    q_next_vanilla = next_target.max(1).values
    y_vanilla = r + GAMMA * (1.0 - done) * q_next_vanilla

target_df = pd.DataFrame({
    "quantity": ["Q_online(s', left)", "Q_online(s', right)", "selected action", "Q_target(s', selected)", "Double target y", "Vanilla max target y"],
    "value": [float(next_online[0,0]), float(next_online[0,1]), int(next_a.item()),
              float(q_next_double.item()), float(y_double.item()), float(y_vanilla.item())],
})
display(target_df.round(4))
print("TD error for selected transition:", round(float((q_sa - y_double).item()), 4))

### Step 13 — Dueling aggregation inside the network

Now open the dueling head for the same state. The torso produces hidden features; the value head says how good the state is overall, the advantage head compares actions, and centering the advantages makes the two action Q-values.

In [ ]:
with torch.no_grad():
    h = online_q.torso(s)
    v = online_q.value_head(h)
    adv = online_q.adv_head(h)
    centered = adv - adv.mean(dim=1, keepdim=True)
    q_from_parts = v + centered
parts_df = pd.DataFrame({
    "action": ["left (0)", "right (1)"],
    "V(s)": [float(v.item()), float(v.item())],
    "A(s,a)": adv.numpy()[0],
    "A - mean(A)": centered.numpy()[0],
    "Q(s,a)": q_from_parts.numpy()[0],
})
display(parts_df.round(4))
fig, axs = plt.subplots(1, 3, figsize=(10, 3))
axs[0].bar(["V"], [float(v.item())], color="#c89bff"); axs[0].set_title("value")
axs[1].bar(parts_df["action"], parts_df["A - mean(A)"], color="#ffb86b"); axs[1].set_title("centered A")
axs[2].bar(parts_df["action"], parts_df["Q(s,a)"], color="#7ee787"); axs[2].set_title("Q = V + A")
for ax in axs: ax.axhline(0, color="k", lw=0.5)
plt.tight_layout(); plt.show()

**🎛️ Play with it — dueling V/A decomposition**

For two actions, the raw advantages are only relative after we subtract their mean. **Try:** change `V`, `A_left`, and `A_right`. **Watch:** raising both advantages equally does not change Q after centering. **Why it matters:** this lets the network learn state quality separately from action preference.

In [ ]:
def play(V=1.0, A_left=0.6, A_right=-0.2):
    A = np.array([A_left, A_right], dtype=float)
    centered = A - A.mean()
    Q = V + centered
    df = pd.DataFrame({"action": ["left", "right"], "A": A, "A - mean(A)": centered, "Q": Q})
    display(df.round(3))
    fig, axs = plt.subplots(1, 2, figsize=(8, 3))
    axs[0].bar(df["action"], df["A - mean(A)"], color="#ffb86b"); axs[0].axhline(0, color="k", lw=0.5)
    axs[0].set_title("centered advantages")
    axs[1].bar(df["action"], df["Q"], color="#7ee787"); axs[1].axhline(V, color="#c89bff", ls="--", label="V")
    axs[1].set_title("Q values"); axs[1].legend()
    plt.show()

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox, Text
    interact(play,
             V=FloatSlider(value=1.0, min=-2.0, max=3.0, step=0.1),
             A_left=FloatSlider(value=0.6, min=-3.0, max=3.0, step=0.1),
             A_right=FloatSlider(value=-0.2, min=-3.0, max=3.0, step=0.1))
except Exception:
    play()        # no ipywidgets (or non-notebook) -> render sensible defaults

**🎛️ Play with it — Double DQN vs vanilla max**

The max over noisy estimates tends to be optimistic. **Try:** make one target value too large, then toggle Double DQN. **Watch:** Double DQN can choose using one set of estimates and evaluate using another. **Why it matters:** separating selection and evaluation reduces overestimation bias.

In [ ]:
def play(use_double=True, online_left=1.0, online_right=1.2, target_left=1.1, target_right=2.0):
    online = np.array([online_left, online_right], dtype=float)
    target = np.array([target_left, target_right], dtype=float)
    if use_double:
        chosen = int(np.argmax(online))
        q_next = target[chosen]
        mode = "Double: online selects, target evaluates"
    else:
        chosen = int(np.argmax(target))
        q_next = target[chosen]
        mode = "Vanilla: target max selects and evaluates"
    y = 1.0 + GAMMA * q_next
    df = pd.DataFrame({"action": ["left", "right"], "online Q(s')": online, "target Q(s')": target})
    display(df.round(3))
    plt.figure(figsize=(6, 3))
    x = np.arange(2)
    plt.bar(x - 0.18, online, width=0.36, label="online", color="#79c0ff")
    plt.bar(x + 0.18, target, width=0.36, label="target", color="#ffb86b")
    plt.xticks(x, ["left", "right"]); plt.ylabel("Q")
    plt.title("next-state estimates")
    plt.legend(); plt.show()
    print(mode)
    print(f"chosen action = {chosen}; target y = 1 + gamma*{q_next:.2f} = {y:.3f}")

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox, Text
    interact(play,
             use_double=Checkbox(value=True),
             online_left=FloatSlider(value=1.0, min=-1, max=3, step=0.1),
             online_right=FloatSlider(value=1.2, min=-1, max=3, step=0.1),
             target_left=FloatSlider(value=1.1, min=-1, max=3, step=0.1),
             target_right=FloatSlider(value=2.0, min=-1, max=3, step=0.1))
except Exception:
    play()        # no ipywidgets (or non-notebook) -> render sensible defaults

### Step 14 — Define one learning update

`learn` performs a single gradient step (COMPONENT 1 base loop + COMPONENT 2 Double-DQN target). It samples a batch, computes the online network's Q for the actions taken, builds the bootstrap target, weights squared TD error by importance-sampling weights, and refreshes priorities.

In [ ]:
# ============================================================================
# COMPONENT 1 (base DQN loop) + COMPONENT 2 (Double DQN target).
# One update: build the target, weight the squared TD loss by IS weights,
# step, and refresh priorities.
# ============================================================================
def learn(q, q_target, opt, replay, batch=64, beta=0.4, use_double=True):
    if len(replay) < batch:
        return
    s, a, r, s2, done, idx, w = replay.sample(batch, beta=beta)
    q_sa = q(s).gather(1, a.unsqueeze(1)).squeeze(1)        # Q_online(s,a) for the action taken
    with torch.no_grad():
        if use_double:                                     # DOUBLE: online selects, target evaluates
            next_a = q(s2).argmax(1, keepdim=True)         # argmax_a Q_online(s2, a)
            q_next = q_target(s2).gather(1, next_a).squeeze(1)
        else:                                              # plain DQN: target net selects AND evaluates
            q_next = q_target(s2).max(1).values
        y = r + GAMMA * (1.0 - done) * q_next              # TD target (1-done zeros terminal bootstrap)
    td = q_sa - y                                          # TD error
    loss = (w * td.pow(2)).mean()                          # IS-weighted squared TD loss
    opt.zero_grad()
    loss.backward()
    opt.step()
    replay.update_priorities(idx, td.detach().cpu().numpy())

### Step 15 — One cheap learning step demo

Before the full run, fill a small replay buffer with random experience and take one update. This is not meant to solve CartPole; it simply proves the update changes parameters and priorities.

In [ ]:
def fill_replay_random(replay, n=80, seed=10):
    env = gym.make("CartPole-v1")
    s, _ = env.reset(seed=seed)
    for t in range(n):
        a = env.action_space.sample()
        s2, r, term, trunc, _ = env.step(a)
        done = term or trunc
        replay.push(s, a, r, s2, float(done))
        s = s2
        if done:
            s, _ = env.reset(seed=seed + t + 1)
    env.close()

torch.manual_seed(0); np.random.seed(0); random.seed(0)
step_q = QNet(4, 2, dueling=True)
step_target = QNet(4, 2, dueling=True)
step_target.load_state_dict(step_q.state_dict())
step_opt = torch.optim.Adam(step_q.parameters(), lr=1e-3)
step_replay = PrioritizedReplay(capacity=200, per_on=True)
fill_replay_random(step_replay, n=80)
probe = torch.tensor(step_replay.buf[0][0], dtype=torch.float32).unsqueeze(0)
with torch.no_grad():
    before_q = step_q(probe).numpy()[0]
learn(step_q, step_target, step_opt, step_replay, batch=32, beta=0.4, use_double=True)
with torch.no_grad():
    after_q = step_q(probe).numpy()[0]
change_df = pd.DataFrame({"action": [0, 1], "before Q": before_q, "after one learn()": after_q})
display(change_df.round(5))
print("first five updated priorities:", np.round(step_replay.prios[:5], 4).tolist())

**🎛️ Play with it — replay size and diversity**

A larger replay buffer can hold experience from more episodes and epsilon values. **Try:** grow the buffer size. **Watch:** the conceptual mix of recent vs older transitions change. **Why it matters:** replay breaks correlations, but a tiny buffer forgets too quickly.

In [ ]:
def play(buffer_size=5000):
    recent = min(buffer_size, 500)
    older = max(buffer_size - recent, 0)
    df = pd.DataFrame({"source": ["most recent", "older diverse"], "slots": [recent, older]})
    display(df)
    plt.figure(figsize=(5, 3))
    plt.bar(df["source"], df["slots"], color=["#79c0ff", "#c89bff"])
    plt.ylabel("stored transitions")
    plt.title("replay diversity")
    plt.show()
    print(f"A capacity of {buffer_size:,} can store about {buffer_size/500:.1f} full 500-step CartPole episodes.")

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox, Text
    interact(play, buffer_size=IntSlider(value=5000, min=500, max=50000, step=500))
except Exception:
    play()        # no ipywidgets (or non-notebook) -> render sensible defaults

**🎛️ Play with it — target-network sync period**

The target network is a delayed copy of the online network. **Try:** change `sync_every`. **Watch:** shorter periods track the online network closely; longer periods make targets more stable but staler. **Why it matters:** DQN needs targets that move slowly enough for bootstrapping to behave.

In [ ]:
def play(sync_every=200, total_steps=1000):
    steps = np.arange(0, total_steps + 1)
    last_sync = (steps // sync_every) * sync_every
    age = steps - last_sync
    plt.figure(figsize=(7, 3))
    plt.plot(steps, age, color="#ffb86b")
    plt.xlabel("environment step"); plt.ylabel("target age (steps)")
    plt.title("target network staleness")
    plt.show()
    print(f"With sync_every={sync_every}, the target copy is at most {sync_every-1} steps old.")

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox, Text
    interact(play,
             sync_every=IntSlider(value=200, min=25, max=500, step=25),
             total_steps=IntSlider(value=1000, min=200, max=2000, step=100))
except Exception:
    play()        # no ipywidgets (or non-notebook) -> render sensible defaults

### Step 16 — Assemble the agent and race it against the baseline

`train` wires the pieces together on CartPole: online net, target net, epsilon-greedy exploration, replay, beta annealing, and periodic target sync. The three toggles switch the improvements on or off. We record **per-episode return** and the moving average so the result plots reuse history instead of retraining.

In [ ]:
# ============================================================================
# Train the assembled agent on CartPole. Toggles select which improvements
# are active. The base target network is always on (it IS the DQN baseline).
# ============================================================================
def train(use_double=True, use_dueling=True, use_per=True,
          episodes=400, sync_every=200, label=""):
    torch.manual_seed(0)
    random.seed(0)
    np.random.seed(0)
    env = gym.make("CartPole-v1")
    obs_dim = env.observation_space.shape[0]
    n_act = env.action_space.n
    q        = QNet(obs_dim, n_act, dueling=use_dueling)
    q_target = QNet(obs_dim, n_act, dueling=use_dueling)
    q_target.load_state_dict(q.state_dict())               # target net starts as a copy
    opt = torch.optim.Adam(q.parameters(), lr=1e-3)
    replay = PrioritizedReplay(per_on=use_per)
    eps, step, recent, history = 1.0, 0, [], []
    print(f"[{label}] double={use_double} dueling={use_dueling} per={use_per}")
    for ep in range(400):
        if ep >= episodes:
            break
        beta = min(1.0, 0.4 + 0.6 * ep / episodes)         # anneal IS exponent beta 0.4 -> 1.0
        s, _ = env.reset(seed=ep)
        done = False
        ep_ret = 0.0
        while not done:
            if random.random() < eps:                      # epsilon-greedy exploration
                a = env.action_space.sample()
            else:
                with torch.no_grad():
                    a = int(q(torch.tensor(s, dtype=torch.float32).unsqueeze(0)).argmax())
            s2, rew, term, trunc, _ = env.step(a)
            done = term or trunc
            replay.push(s, a, rew, s2, float(done))
            s = s2
            ep_ret += rew
            step += 1
            learn(q, q_target, opt, replay, beta=beta, use_double=use_double)
            if step % sync_every == 0:                     # periodic target-net sync
                q_target.load_state_dict(q.state_dict())
        eps = max(0.02, eps * 0.97)                         # decay exploration
        recent.append(ep_ret)
        avg = sum(recent[-100:]) / len(recent[-100:])
        history.append({"episode": ep, "return": ep_ret, "moving_avg": avg,
                        "epsilon": eps, "label": label})
        if ep % 20 == 0:
            print(f"  ep {ep:3d}  eps {eps:.2f}  avg return (last 100): {avg:6.1f}")
        if len(recent) >= 100 and avg >= 475:
            print(f"  -> SOLVED CartPole at episode {ep} (avg return {avg:.1f} >= 475).")
            break
    env.close()
    return pd.DataFrame(history)

### Step 17 — Final run: Rainbow-lite vs vanilla DQN

This is the real comparison from our run. Both agents use the same net size, optimizer, learning rate, epsilon schedule, seed, and environment. Only the three improvement toggles differ.

In [ ]:
# --- FINAL RUN: combined "Rainbow-lite" agent vs the vanilla DQN baseline. ---
print("\n=== Combined agent (Double + Dueling + Prioritized replay) ===")
combined_hist = train(use_double=True, use_dueling=True, use_per=True, label="Rainbow-lite")

print("\n=== Vanilla DQN baseline (all three toggles OFF) ===")
vanilla_hist  = train(use_double=False, use_dueling=False, use_per=False, label="Vanilla DQN")

print("\nCombined avg-return trajectory:", [round(h, 1) for h in combined_hist["moving_avg"].iloc[::20].tolist()])
print("Vanilla  avg-return trajectory:", [round(h, 1) for h in vanilla_hist["moving_avg"].iloc[::20].tolist()])
# The combined agent climbs to ~500 and SOLVES CartPole noticeably faster and more stably
# than the vanilla DQN. Flip any single toggle to isolate that improvement's effect.
# (Exact numbers vary by hardware/seed; our small run, NOT the paper's Atari results.)

## Visualize the data & results

_Does combining all four improvements — DQN base + Double-DQN target + Dueling head + Prioritized replay — make the agent solve CartPole faster and more stably than a vanilla DQN?_ We answer with the per-episode histories we just recorded — no retraining needed.

### Step 18 — Return curves with the 475 solved line

Plot both raw episode returns (light) and the moving average (bold). CartPole-v1 is considered solved when the 100-episode average reaches 475.

In [ ]:
results = pd.concat([combined_hist, vanilla_hist], ignore_index=True)
plt.figure(figsize=(8, 4))
colors = {"Rainbow-lite": "#7ee787", "Vanilla DQN": "#79c0ff"}
for label, grp in results.groupby("label"):
    plt.plot(grp["episode"], grp["return"], color=colors[label], alpha=0.25, lw=1)
    plt.plot(grp["episode"], grp["moving_avg"], color=colors[label], lw=2.2, label=label)
plt.axhline(475, color="#ff6b6b", ls="--", label="solved = 475")
plt.xlabel("episode"); plt.ylabel("return")
plt.title("CartPole returns")
plt.legend(); plt.show()

### Step 19 — Summary table and final-return bar chart

The table records final moving average, best moving average, and the first episode where the moving average crosses 475 if it happened in our run.

In [ ]:
summary_rows = []
for label, grp in results.groupby("label"):
    solved = grp.loc[grp["moving_avg"] >= 475, "episode"]
    summary_rows.append({
        "agent": label,
        "episodes run": int(len(grp)),
        "final moving avg": float(grp["moving_avg"].iloc[-1]),
        "best moving avg": float(grp["moving_avg"].max()),
        "first solved episode": None if solved.empty else int(solved.iloc[0]),
    })
summary = pd.DataFrame(summary_rows)
display(summary.round(1))
plt.figure(figsize=(6, 3))
plt.bar(summary["agent"], summary["final moving avg"], color=[colors[a] for a in summary["agent"]])
plt.axhline(475, color="#ff6b6b", ls="--")
plt.ylabel("final moving avg")
plt.title("final comparison")
plt.show()

### Step 20 — What the trained policy prefers for one state

Use the final combined agent's architecture and a short retraining-free probe would require returning the model. To keep the public `train` API simple, we instead visualize the already-built `online_q` hero model's action scores for the sampled state. The trained-result evidence is the recorded return curve above.

In [ ]:
with torch.no_grad():
    hero_scores = online_q(s).numpy()[0]
policy_df = pd.DataFrame({"action": ["left", "right"], "Q score from hero model": hero_scores})
display(policy_df.round(4))
plt.figure(figsize=(5, 3))
plt.bar(policy_df["action"], policy_df["Q score from hero model"], color="#c89bff")
plt.ylabel("Q score")
plt.title("sample-state action scores")
plt.show()

### Practice

1. Flip only `use_double=True` while leaving `use_dueling=False` and `use_per=False`. Does Double DQN alone help in your run?
2. Flip only `use_dueling=True`. Inspect whether the value stream and advantage stream learn different-looking outputs.
3. Flip only `use_per=True`. Try `alpha=0.0` and `alpha=0.9` in `PrioritizedReplay` and compare the curves.
4. Change `sync_every`. What happens when the target network is copied too often or too rarely?
5. CartPole is small; try a smaller hidden size (for example 64). Does Rainbow-lite still solve it?